### 06_Competitive_Audit_Benchmarking.ipynb

**Purpose:**  
To benchmark the performance of baseline sentiment analysis (VADER) across four competing fintech apps: Venmo, Cash App, Chime, and PayPal. This notebook moves from general model evaluation to competitive intelligence by quantifying the "Hidden Negative" problem for each specific platform.

**Key Questions Answered:**
1. Which app has the highest rate of hidden negative reviews?
2. Which app has the most severe complaints being missed by baseline tools?
3. Where are the competitive gaps in product reliability (Critical vs. High severity misses)?

**Outcome:**  
Identification of specific competitive weaknesses and proof that domain-specific NLP is required to capture the true risk levels of fintech customer dissatisfaction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option('display.max_colwidth', 120)
sns.set_style('whitegrid')

df = pd.read_csv("../data/clean/all_apps_clean.csv")
analyzer = SentimentIntensityAnalyzer()

def get_vader_score(text):
    if pd.isna(text): return 0.0
    return analyzer.polarity_scores(str(text))['compound']

def vader_label(compound):
    if compound >= 0.05: return 'Positive'
    elif compound <= -0.05: return 'Negative'
    else: return 'Neutral'

def rating_label(rating):
    if rating >= 4: return 'Positive'
    elif rating <= 2: return 'Negative'
    else: return 'Neutral'

severity_keywords = {
    5: ['locked out', 'account locked', 'account closed', 'money missing',
        'funds missing', 'stolen', 'fraud', 'scam', 'cannot access',
        'lost my money', 'money gone', 'unauthorized'],
    4: ['declined', 'card declined', 'transfer failed', 'verification failed',
        'failed', 'frozen', 'pending', 'verification', "can't send",
        "can't receive", 'not working', 'blocked'],
    3: ['customer service', 'support', 'refund', 'delay', 'late',
        'problem', 'issue', 'error', 'wrong', 'charge'],
    2: ['slow', 'bug', 'glitch', 'annoying', 'confusing', 'difficult'],
    1: ['minor', 'small', 'slight', 'okay', 'fine']
}

def compute_severity(text):
    if pd.isna(text): return 1
    text = str(text).lower()
    for level in [5, 4, 3, 2, 1]:
        for kw in severity_keywords[level]:
            if kw in text:
                return level
    return 1

severity_map = {1:'Low', 2:'Low-Moderate', 3:'Moderate', 4:'High', 5:'Critical'}

df['vader_compound']   = df['review_clean'].apply(get_vader_score)
df['vader_sentiment']  = df['vader_compound'].apply(vader_label)
df['rating_sentiment'] = df['rating'].apply(rating_label)
df['severity_score']   = df['review_clean'].apply(compute_severity)
df['severity_label']   = df['severity_score'].map(severity_map)

hidden_neg = df[
    (df['rating_sentiment'] == 'Negative') &
    (df['vader_sentiment']  != 'Negative')
].copy()

print(f"✅ Setup complete. df shape: {df.shape}")
print(f"✅ Hidden negatives: {len(hidden_neg):,}")

SO WHAT WE ARE DOING HERE IS TO CLACULATE HOW BADLY VADER FAILED TO CATCH NEGATIVE REVIEWS, AND HOW SEVERE THOSE MISSED REVIEWS WERE

In [ ]:
# Build competitive benchmarking table by app

#we are creating an empty list. We will collect one dictionary per app and store it here. AT the end we turn it into a dataframe for display and sorting.
benchmarks = []



for app in df['app_name'].unique(): #we loop through each unique app name in the dataset. So it runs 4 times - once for each app.
    app_df = df[df['app_name'] == app] #Filters the full dataframe down to only rows for that app. So on the Venmo loop, app_df is only Venmo reviews
    
    true_neg = app_df[app_df['rating_sentiment'] == 'Negative']
    hidden = true_neg[true_neg['vader_sentiment'] != 'Negative']
    
    benchmarks.append({
        'App': app,
        'Total Reviews': len(app_df),
        'True Negatives': len(true_neg),
        'Hidden Negatives': len(hidden),
        'Hidden Neg Rate': round(len(hidden) / len(true_neg), 3) if len(true_neg) > 0 else 0,
        'Avg Hidden Severity': round(hidden['severity_score'].mean(), 2),
        'Critical Misses': (hidden['severity_score'] == 5).sum(),
        'High Misses': (hidden['severity_score'] == 4).sum(),
    })

benchmark_df = pd.DataFrame(benchmarks).sort_values('Hidden Neg Rate', ascending=False)
benchmark_df['Hidden Neg Rate'] = benchmark_df['Hidden Neg Rate'].map(lambda x: f"{x:.1%}")

print("=== COMPETITIVE BENCHMARKING TABLE ===")
display(benchmark_df)

NOW LET US VISUALIZE THE BENCHMARKING RESULTS

In [ ]:
plot_df = pd.DataFrame(benchmarks).sort_values('Hidden Neg Rate', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1: Hidden negative rate by app
axes[0].bar(plot_df['App'], plot_df['Hidden Neg Rate'], color='#c0392b', edgecolor='black')
axes[0].set_title("Hidden Negative Rate\nby App", fontsize=13)
axes[0].set_ylabel("Rate")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0].tick_params(axis='x', rotation=20)

# Chart 2: Average hidden severity by app
axes[1].bar(plot_df['App'], plot_df['Avg Hidden Severity'], color='#e67e22', edgecolor='black')
axes[1].set_title("Avg Severity of Hidden\nNegatives by App", fontsize=13)
axes[1].set_ylabel("Avg Severity Score (1-5)")
axes[1].tick_params(axis='x', rotation=20)

# Chart 3: Critical + High misses stacked
axes[2].bar(plot_df['App'], plot_df['Critical Misses'], color='#922b21', edgecolor='black', label='Critical')
axes[2].bar(plot_df['App'], plot_df['High Misses'], bottom=plot_df['Critical Misses'], color='#e74c3c', edgecolor='black', label='High')
axes[2].set_title("Critical + High Severity\nMisses by App", fontsize=13)
axes[2].set_ylabel("Count")
axes[2].legend()
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle("Competitive Intelligence: VADER Failure Analysis by App", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("competitive_benchmarking.png", dpi=150, bbox_inches='tight')
plt.show()

COMPETITIVE GAP HEATMAP (THE "EXECUTIVE" CHART)

In [ ]:
# Build severity bucket counts per app
severity_order = ['Critical', 'High', 'Moderate', 'Low-Moderate', 'Low']

heatmap_data = (
    hidden_neg.groupby(['app_name', 'severity_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=severity_order, fill_value=0)
)

plt.figure(figsize=(10, 5))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='d',
    cmap='Reds',
    linewidths=0.5,
    linecolor='gray'
)
plt.title("Competitive Gap Heatmap\nHidden Negative Counts by App and Severity", fontsize=14, fontweight='bold')
plt.xlabel("Severity")
plt.ylabel("App")
plt.tight_layout()
plt.savefig("competitive_gap_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()